# Eixo 3: Economia e Agropecuária

## Fontes de Dados
- **PAM (IBGE)**: Produção Agrícola Municipal: área plantada e valor da produção
- **PPM (IBGE)**: Pesquisa da Pecuária Municipal: evolução do rebanho bovino
- **PIB Municipal (IBGE)**: Valor Adicionado Bruto (VAB) exclusivo da Agropecuária
- **Comex Stat (MDIC)**: Volume e valor de exportação (soja, carne) por município

---

## Análise 1: Índice de Custo Ambiental (ICA)

### Especificação BDD

**Feature**: Identificar municípios com pior relação custo-benefício ecológico

**Scenario**: Calcular o Índice de Custo Ambiental para cada município

- **GIVEN** que tenho dados de desmatamento (PRODES) por município em hectares
- **AND** que tenho dados de Valor Adicionado Bruto Agropecuário (PIB) por município em R$
- **WHEN** eu calculo o delta (variação) de desmatamento entre dois períodos (ex: 2010-2020)
- **AND** calculo o delta de VAB Agropecuário no mesmo período
- **AND** aplico a fórmula: ICA = ΔDesmatamento (ha) / ΔVAB_Agro (R$)
- **AND** trato casos de divisão por zero adicionando pequena constante ao denominador
- **AND** aplico transformação logarítmica para reduzir influência de outliers
- **THEN** devo obter um ranking de municípios por ICA
- **AND** municípios com ICA alto são ineficientes (muito desmatamento, pouco ganho econômico)
- **AND** municípios com ICA baixo são eficientes (pouco desmatamento, muito ganho econômico)
- **AND** devo identificar os top 50 municípios com pior ICA

### Matemática em Linguagem de Negócio

**1. Delta (Variação):**
- Delta = Valor Final - Valor Inicial
- Delta Desmatamento = Desmatamento_2020 - Desmatamento_2010
- Delta VAB = VAB_2020 - VAB_2010
- Delta positivo = aumento, Delta negativo = redução

**2. Fórmula do ICA:**
- ICA = ΔDesmatamento (ha) / ΔVAB_Agro (R$)
- Mede hectares desmatados por cada R$ 1 de VAB gerado
- ICA alto = muito desmatamento para pouco ganho econômico (ineficiente)
- ICA baixo = pouco desmatamento para muito ganho econômico (eficiente)

**3. Tratamento de Divisão por Zero:**
- Se ΔVAB = 0 ou muito próximo de 0, adicionar pequena constante (ex: 1)
- ICA ajustado = ΔDesmatamento / (ΔVAB + 1)
- Isso evita divisão por zero e valores extremos

**4. Transformação Logarítmica:**
- Log(ICA) = Log(ΔDesmatamento) - Log(ΔVAB)
- Reduz influência de outliers (municípios com valores extremos)
- Torna distribuição mais normal (gaussiana)
- Facilita comparação e identificação de padrões

**5. Interpretação de Negócio:**
- ICA > 0.01: mais de 0.01 ha desmatado por R$ 1 gerado (muito ineficiente)
- ICA entre 0.001 e 0.01: ineficiência moderada
- ICA < 0.001: menos de 0.001 ha desmatado por R$ 1 gerado (eficiente)
- ICA negativo: desmatamento diminuiu enquanto economia cresceu (ideal)

**6. Limitações da Fórmula:**
- Não considera tempo de maturação (lag entre desmatamento e produção)
- Não controla por variáveis de confusão (infraestrutura, clima, qualidade do solo)
- Pode ser enviesado por municípios com VAB pequeno (viés de escala)
- Assume causalidade direta (desmatamento → VAB), mas pode haver causalidade reversa

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configurações
DATA_DIR = Path("/app/data")

# Carregar dados Silver (já consolidados)
pib_path = DATA_DIR / "02_silver/pib_vab_consolidado.parquet"
df_pib = pd.read_parquet(pib_path)

print("Dados PIB VAB Agropecuário carregados:")
print(f"Shape: {df_pib.shape}")
print(f"\nColunas: {df_pib.columns.tolist()}")
print(f"\nPrimeiras linhas:")
df_pib.head()

In [ ]:
# Carregar dados PRODES agregados por município
# Nota: Se não houver dados agregados, fazer agregação a partir dos dados brutos
prodes_path = DATA_DIR / "01_bronze/prodes/prodes_desmatamento_anual.parquet"
df_prodes = pd.read_parquet(prodes_path)

# Converter km² para hectares
df_prodes['desmatamento_ha'] = df_prodes['desmatamento_km2'] * 100

print("Dados PRODES carregados:")
print(f"Shape: {df_prodes.shape}")
print(f"\nColunas: {df_prodes.columns.tolist()}")
print(f"\nPrimeiras linhas:")
df_prodes.head()

In [ ]:
# Definir período de análise (ex: 2010 a 2020)
ano_inicial = 2010
ano_final = 2020

# Calcular delta de desmatamento por município
desmatamento_inicial = df_prodes[df_prodes['ano'] == ano_inicial].groupby('codigo_ibge')['desmatamento_ha'].sum()
desmatamento_final = df_prodes[df_prodes['ano'] == ano_final].groupby('codigo_ibge')['desmatamento_ha'].sum()

delta_desmatamento = desmatamento_final - desmatamento_inicial
delta_desmatamento = delta_desmatamento.reset_index()
delta_desmatamento.columns = ['codigo_ibge', 'delta_desmatamento_ha']

print("Delta de desmatamento por município:")
print(f"Período: {ano_inicial} a {ano_final}")
delta_desmatamento.head()

In [ ]:
# Calcular delta de VAB Agropecuário por município
vab_inicial = df_pib[df_pib['ano'] == ano_inicial].groupby('codigo_ibge')['vab_agropecuario'].sum()
vab_final = df_pib[df_pib['ano'] == ano_final].groupby('codigo_ibge')['vab_agropecuario'].sum()

delta_vab = vab_final - vab_inicial
delta_vab = delta_vab.reset_index()
delta_vab.columns = ['codigo_ibge', 'delta_vab_agro']

print("Delta de VAB Agropecuário por município:")
print(f"Período: {ano_inicial} a {ano_final}")
delta_vab.head()

In [ ]:
# Merge dos deltas
df_ica = pd.merge(delta_desmatamento, delta_vab, on='codigo_ibge', how='inner')

# Tratar valores nulos
df_ica = df_ica.dropna()

# Filtrar apenas deltas positivos de VAB (crescimento econômico)
# ICA só faz sentido quando houve crescimento econômico
df_ica = df_ica[df_ica['delta_vab_agro'] > 0]

# Calcular ICA com tratamento de divisão por zero
df_ica['ica'] = df_ica['delta_desmatamento_ha'] / (df_ica['delta_vab_agro'] + 1)

# Calcular ICA logarítmico
df_ica['ica_log'] = np.log(df_ica['delta_desmatamento_ha'] + 1) - np.log(df_ica['delta_vab_agro'] + 1)

print("Índice de Custo Ambiental (ICA) por município:")
df_ica.head()

In [ ]:
# Identificar top 50 municípios com pior ICA (mais ineficientes)
top_50_ica = df_ica.nlargest(50, 'ica')

print("Top 50 Municípios com Pior ICA (Mais Ineficientes):")
print("(Muito desmatamento para pouco ganho econômico)")
top_50_ica[['codigo_ibge', 'delta_desmatamento_ha', 'delta_vab_agro', 'ica']].head(10)

In [ ]:
# Estatísticas descritivas do ICA
print("Estatísticas Descritivas do ICA:")
print(df_ica['ica'].describe())

# Distribuição do ICA
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(df_ica['ica'], bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('ICA', fontsize=12)
plt.ylabel('Frequência', fontsize=12)
plt.title('Distribuição do ICA', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.hist(df_ica['ica_log'], bins=50, edgecolor='black', alpha=0.7, color='orange')
plt.xlabel('ICA (escala log)', fontsize=12)
plt.ylabel('Frequência', fontsize=12)
plt.title('Distribuição do ICA (Log)', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: Delta Desmatamento vs Delta VAB
plt.figure(figsize=(10, 8))

plt.scatter(df_ica['delta_vab_agro'], 
            df_ica['delta_desmatamento_ha'], 
            alpha=0.5, s=20)

# Destacar top 50 piores ICA
plt.scatter(top_50_ica['delta_vab_agro'], 
            top_50_ica['delta_desmatamento_ha'], 
            color='red', alpha=0.7, s=50, label='Top 50 Piores ICA')

plt.xlabel('Delta VAB Agropecuário (R$)', fontsize=12)
plt.ylabel('Delta Desmatamento (ha)', fontsize=12)
plt.title('Eficiência Econômica: Desmatamento vs VAB', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

# Adicionar linha de eficiência média (ICA médio)
ica_medio = df_ica['ica'].median()
x_line = np.linspace(df_ica['delta_vab_agro'].min(), df_ica['delta_vab_agro'].max(), 100)
y_line = ica_medio * x_line
plt.plot(x_line, y_line, 'g--', label=f'ICA Médio = {ica_medio:.6f}', alpha=0.7)

plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

### Conclusão da Análise 1

**Interpretação de Negócio:**
- O ICA médio é [valor], indicando que em média [X] hectares são desmatados por R$ 1 de VAB gerado
- Os top 50 municípios com pior ICA são [proporção]% mais ineficientes que a média
- Isso sugere que [insights sobre eficiência econômica]

**Limitações Metodológicas:**
- Não considera lag temporal entre desmatamento e produção
- Pode ser enviesado por variáveis omitidas (infraestrutura, clima)
- Viés de escala para municípios pequenos

**Recomendações:**
- Implementar modelo de diferenças-em-diferenças para causalidade
- Controlar por variáveis de confusão
- Usar ICA relativo (ranking percentual) para comparabilidade

---
## Análise 2: Eficiência da Pecuária vs Agricultura

### Especificação BDD

**Feature**: Comparar eficiência econômica entre pecuária e agricultura

**Scenario**: Calcular rendimento por hectare desmatado para cada setor

- **GIVEN** que tenho dados de rebanho bovino (PPM) por município
- **AND** que tenho dados de área plantada e valor da produção (PAM) por município
- **AND** que tenho dados de desmatamento (PRODES) por município
- **WHEN** eu classifico municípios como "dominantes em pecuária" ou "dominantes em agricultura"
- **AND** calculo o delta de rebanho e delta de área plantada
- **AND** calculo o delta de valor da produção para cada setor
- **AND** calculo o rendimento: R$ gerado por hectare desmatado
- **THEN** devo obter o rendimento médio da pecuária vs agricultura
- **AND** devo identificar qual setor é mais eficiente economicamente
- **AND** devo calcular a correlação entre crescimento do setor e desmatamento

### Matemática em Linguagem de Negócio

**1. Classificação de Dominância:**
- Município dominante em pecuária: delta rebanho > delta área plantada
- Município dominante em agricultura: delta área plantada > delta rebanho
- Município misto: deltas similares (diferença < 20%)

**2. Rendimento por Hectare Desmatado:**
- Rendimento Pecuária = ΔValor Produção Pecuária / ΔDesmatamento
- Rendimento Agricultura = ΔValor Produção Agricultura / ΔDesmatamento
- Mede R$ gerado por cada hectare desmatado
- Rendimento alto = setor economicamente eficiente
- Rendimento baixo = setor economicamente ineficiente

**3. Correlação:**
- Correlação Pearson entre delta rebanho e delta desmatamento
- Correlação Pearson entre delta área plantada e delta desmatamento
- r > 0.7: correlação forte (setor impulsiona desmatamento)
- r < 0.3: correlação fraca (setor não é principal driver)

**4. Interpretação de Negócio:**
- Pecuária mais eficiente: pecuária gera mais R$/ha desmatado
- Agricultura mais eficiente: agricultura gera mais R$/ha desmatado
- Se pecuária tem baixa eficiência mas alta correlação: pecuária é vetor de baixo valor agregado
- Se agricultura tem alta eficiência e alta correlação: agricultura é vetor de alto valor agregado

**5. Limitações:**
- Classificação binária ignora sistemas mistos predominantes
- Não considera intensificação (ex: confinamento vs pastejo extensivo)
- Viés de sobrevivência (municípios que já desmataram podem ter transicionado)

In [ ]:
# Carregar dados PPM (Pecuária)
ppm_path = DATA_DIR / "02_silver/ppm_consolidado.parquet"
df_ppm = pd.read_parquet(ppm_path)

# Carregar dados PAM (Agricultura)
pam_path = DATA_DIR / "02_silver/pam_consolidado.parquet"
df_pam = pd.read_parquet(pam_path)

print("Dados PPM carregados:")
print(f"Shape: {df_ppm.shape}")
print(f"\nDados PAM carregados:")
print(f"Shape: {df_pam.shape}")

In [ ]:
# Calcular delta de rebanho bovino por município
rebanho_inicial = df_ppm[df_ppm['ano'] == ano_inicial].groupby('codigo_ibge')['rebanho_bovino'].sum()
rebanho_final = df_ppm[df_ppm['ano'] == ano_final].groupby('codigo_ibge')['rebanho_bovino'].sum()
delta_rebanho = (rebanho_final - rebanho_inicial).reset_index()
delta_rebanho.columns = ['codigo_ibge', 'delta_rebanho']

# Calcular delta de área plantada por município
area_inicial = df_pam[df_pam['ano'] == ano_inicial].groupby('codigo_ibge')['area_plantada_ha'].sum()
area_final = df_pam[df_pam['ano'] == ano_final].groupby('codigo_ibge')['area_plantada_ha'].sum()
delta_area = (area_final - area_inicial).reset_index()
delta_area.columns = ['codigo_ibge', 'delta_area_plantada']

# Calcular delta de valor da produção por município
valor_inicial = df_pam[df_pam['ano'] == ano_inicial].groupby('codigo_ibge')['valor_producao'].sum()
valor_final = df_pam[df_pam['ano'] == ano_final].groupby('codigo_ibge')['valor_producao'].sum()
delta_valor = (valor_final - valor_inicial).reset_index()
delta_valor.columns = ['codigo_ibge', 'delta_valor_producao']

print("Deltas calculados por município:")
print(f"Delta rebanho: {len(delta_rebanho)} municípios")
print(f"Delta área plantada: {len(delta_area)} municípios")
print(f"Delta valor produção: {len(delta_valor)} municípios")

In [ ]:
# Merge de todos os deltas
df_eficiencia = pd.merge(delta_desmatamento, delta_rebanho, on='codigo_ibge', how='inner')
df_eficiencia = pd.merge(df_eficiencia, delta_area, on='codigo_ibge', how='inner')
df_eficiencia = pd.merge(df_eficiencia, delta_valor, on='codigo_ibge', how='inner')

# Classificar municípios por dominância
df_eficiencia['dominancia'] = df_eficiencia.apply(
    lambda x: 'Pecuária' if x['delta_rebanho'] > x['delta_area_plantada'] else 'Agricultura',
    axis=1
)

# Calcular rendimento por hectare desmatado
# Usando delta de valor da produção como proxy de ganho econômico
df_eficiencia['rendimento_ha'] = df_eficiencia['delta_valor_producao'] / (df_eficiencia['delta_desmatamento_ha'] + 1)

print("Eficiência por município:")
df_eficiencia.head()

In [ ]:
# Calcular rendimento médio por setor
rendimento_por_setor = df_eficiencia.groupby('dominancia')['rendimento_ha'].agg(['mean', 'median', 'std', 'count'])

print("Rendimento por Hectare Desmatado por Setor:")
print("(R$ gerado por hectare desmatado)")
rendimento_por_setor

In [ ]:
# Calcular correlação entre crescimento do setor e desmatamento
corr_pecuaria = df_eficiencia[df_eficiencia['dominancia'] == 'Pecuária']['delta_rebanho'].corr(
    df_eficiencia[df_eficiencia['dominancia'] == 'Pecuária']['delta_desmatamento_ha']
)

corr_agricultura = df_eficiencia[df_eficiencia['dominancia'] == 'Agricultura']['delta_area_plantada'].corr(
    df_eficiencia[df_eficiencia['dominancia'] == 'Agricultura']['delta_desmatamento_ha']
)

print("Correlação entre Crescimento do Setor e Desmatamento:")
print(f"Pecuária: r = {corr_pecuaria:.4f}")
print(f"Agricultura: r = {corr_agricultura:.4f}")

In [ ]:
# Visualizar rendimento por setor
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.boxplot(data=df_eficiencia, x='dominancia', y='rendimento_ha')
plt.yscale('log')  # Escala log devido a outliers
plt.xlabel('Setor Dominante', fontsize=12)
plt.ylabel('Rendimento (R$/ha) - Escala Log', fontsize=12)
plt.title('Rendimento por Setor', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='y')

plt.subplot(1, 2, 2)
setores = ['Pecuária', 'Agricultura']
correlacoes = [corr_pecuaria, corr_agricultura]
cores = ['green' if c > 0.5 else 'orange' if c > 0.3 else 'red' for c in correlacoes]

bars = plt.bar(setores, correlacoes, color=cores)
plt.xlabel('Setor', fontsize=12)
plt.ylabel('Correlação com Desmatamento', fontsize=12)
plt.title('Correlação: Crescimento vs Desmatamento', fontsize=14, fontweight='bold')
plt.axhline(y=0.7, color='green', linestyle='--', alpha=0.5, label='Correlação Forte (>0.7)')
plt.axhline(y=0.3, color='orange', linestyle='--', alpha=0.5, label='Correlação Moderada (0.3-0.7)')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

### Conclusão da Análise 2

**Interpretação de Negócio:**
- O rendimento médio da pecuária é [R$/ha]
- O rendimento médio da agricultura é [R$/ha]
- A correlação da pecuária com desmatamento é [FORTE/MODERADA/FRACA] (r = [valor])
- A correlação da agricultura com desmatamento é [FORTE/MODERADA/FRACA] (r = [valor])

**Conclusão:**
- [Pecuária/Agricultura] é mais eficiente economicamente
- [Pecuária/Agricultura] tem correlação mais forte com desmatamento
- Isso sugere que [pecuária/agricultura] é o principal vetor de desmatamento

---
## Resumo do Eixo 3: Economia e Agropecuária

**Principais Descobertas:**
1. ICA médio: [valor] hectares desmatados por R$ 1 de VAB gerado
2. Eficiência comparativa: [pecuária/agricultura] é mais eficiente
3. Correlação com desmatamento: [setor] tem correlação mais forte

**Implicações de Negócio:**
- [Setor prioritário para intervenção]
- [Políticas recomendadas baseadas em eficiência]

**Limitações Metodológicas:**
- [Listar limitações identificadas]

**Próximos Passos:**
- [Análises adicionais recomendadas]
- [Integração com Eixo 4: Impacto Socioambiental]